In [1]:
import transformers
from transformers import BitsAndBytesConfig
import torch
import pandas as pd
import os

/home/oenni/.venvs/Llama3-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

In [3]:
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
torch.cuda.is_available()

True

In [4]:
pipeline = transformers.pipeline("text-generation", 
                                 model=model_id, 
                                 model_kwargs={"torch_dtype":torch.float16}, 
                                 device_map="auto",
                                )

Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.00s/it]
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [5]:
#messages = [
#        {"role": "system", "content": "You are a classification system trained to classify pairs of statements into 'self-contradiction' and 'non-contradiction'. You can always assume that the statements are uttered by the same speaker. Only return the label."},
#        {"role": "user", "content": "Please classify the following pair: Sentence1:'I love Donald Trump', Sentence2:'I hate Donald Trump'"},
#        ]

messages = [
    {"role": "system", "content": "You are a helpful argumentation assistant that will give an original statement on a specified topic with a specified relation to an argument you receive. You will only generate the statement."},
    {"role": "user", "content": "Please give an original statement on the topic of 'school uniforms' supported by the argument:'People who are for uniforms say that it promotes social conformity , so the less-to-do do not have to be pressured to keep up with their well-off contemporaries .'"},
]

In [6]:
terminators = [
    pipeline.tokenizer.eos_token_id,
    pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = pipeline(
    messages,
    max_new_tokens=200,
    eos_token_id=terminators,
    do_sample=True,
    temperature=0.1,
    top_p=0.9,
)
assistant_response = outputs[0]["generated_text"][-1]["content"]
print(assistant_response)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


"By implementing school uniforms, educators can create a level playing field where students are judged solely on their academic abilities and personal character, rather than their socioeconomic status, thereby reducing the pressure on less-privileged students to conform to the materialistic standards set by their wealthier peers."


In [ ]:
sent_type = "proposition" # "proposition" "locution"

seed = 52 #42, 121, 78, 52

inds = [4]#[2, 8, 16, 32]
preds = [[]] #[[],[],[],[]]

In [ ]:
for i in range(len(inds)):
    print(i, len(preds))
    samp_num = inds[i]
    print(samp_num)
    prime_df = pd.read_csv(f"primes/{sent_type}_primes_{samp_num}_{seed}.tsv", sep="\t")
    sample_df = pd.read_csv(f"samples/{sent_type}_samples_{samp_num}_{seed}.tsv", sep="\t")

    primes = list(prime_df["prime_sample"])
    prime_labels = list(prime_df["label"])

    samples = list(sample_df["test_sample"])
    sample_labels = list(sample_df["label"])
    
    prime_prompt = "To help you, please consider the following examples: \n"
    for prime, label in zip(primes, prime_labels):
        prime_prompt += f"{prime} would be classified as {label} \n"
        
    sample_list = []
    for x in range(len(samples)):
        samp = samples[x].split(",")
        sample_list.append(f"Please classify the following pair: Sentence1: {samp[0]}, Sentence2: {samp[1]}.")

    for j in range(len(sample_list)):
        messages = [
        {"role": "system", "content": f"You are a classification system trained to classify pairs of statements into 'self-contradiction' and 'non-contradiction'. You can always assume that the statements are uttered by the same speaker. Only return the label.{prime_prompt}"},
        {"role": "user", "content": sample_list[j]},
        ]
        prompt = pipeline.tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=True
        )
        terminators = [
        pipeline.tokenizer.eos_token_id,
        pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
        ]
        outputs = pipeline(
        prompt,
        max_new_tokens=10,
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.5,
        top_p=0.9,
        )
        preds[i].append(outputs[0]["generated_text"][len(prompt):])

    out_df = pd.DataFrame()
    out_df["ID"] = sample_df["ID"]
    out_df["sample"] = samples
    out_df["prediction"] = preds[i]
    out_df["gold_label"] = sample_labels
    
    out_df.to_csv(f"Llama3-8B_{sent_type}_{samp_num}_{seed}_1.tsv", sep="\t", index=True)
    

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


0 1
4


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
/usr/local/jupyterhub/lib64/python3.10/site-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:

In [ ]:
preds = [[],[],[],[]]
for i in range(len(inds)):
    print(i, len(preds))
    samp_num = inds[i]
    print(samp_num)
    prime_df = pd.read_csv(f"primes/{sent_type}_primes_{samp_num}_{seed}.tsv", sep="\t")
    sample_df = pd.read_csv(f"samples/{sent_type}_samples_{samp_num}_{seed}.tsv", sep="\t")

    primes = list(prime_df["prime_sample"])
    prime_labels = list(prime_df["label"])

    samples = list(sample_df["test_sample"])
    sample_labels = list(sample_df["label"])
    
    prime_prompt = "To help you, please consider the following examples: \n"
    for prime, label in zip(primes, prime_labels):
        prime_prompt += f"{prime} would be classified as {label} \n"
        
    sample_list = []
    for x in range(len(samples)):
        samp = samples[x].split(",")
        sample_list.append(f"Please classify the following pair: Sentence1: {samp[0]}, Sentence2: {samp[1]}.")

    for j in range(len(sample_list)):
        messages = [
        {"role": "system", "content": f"You are a classification system trained to classify pairs of statements into 'self-contradiction' and 'non-contradiction'. You can always assume that the statements are uttered by the same speaker. Only return the label.{prime_prompt}"},
        {"role": "user", "content": sample_list[j]},
        ]
        prompt = pipeline.tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=True
        )
        terminators = [
        pipeline.tokenizer.eos_token_id,
        pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
        ]
        outputs = pipeline(
        prompt,
        max_new_tokens=10,
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.5,
        top_p=0.9,
        )
        preds[i].append(outputs[0]["generated_text"][len(prompt):])

    out_df = pd.DataFrame()
    out_df["ID"] = sample_df["ID"]
    out_df["sample"] = samples
    out_df["prediction"] = preds[i]
    out_df["gold_label"] = sample_labels
    
    out_df.to_csv(f"Llama3-8B_{sent_type}_{samp_num}_{seed}_2.tsv", sep="\t", index=True)
    

/usr/local/jupyterhub/lib64/python3.10/site-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


0 4
4


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for

In [ ]:
preds = [[],[],[],[]]
for i in range(len(inds)):
    print(i, len(preds))
    samp_num = inds[i]
    print(samp_num)
    prime_df = pd.read_csv(f"primes/{sent_type}_primes_{samp_num}_{seed}.tsv", sep="\t")
    sample_df = pd.read_csv(f"samples/{sent_type}_samples_{samp_num}_{seed}.tsv", sep="\t")

    primes = list(prime_df["prime_sample"])
    prime_labels = list(prime_df["label"])

    samples = list(sample_df["test_sample"])
    sample_labels = list(sample_df["label"])
    
    prime_prompt = "To help you, please consider the following examples: \n"
    for prime, label in zip(primes, prime_labels):
        prime_prompt += f"{prime} would be classified as {label} \n"
        
    sample_list = []
    for x in range(len(samples)):
        samp = samples[x].split(",")
        sample_list.append(f"Please classify the following pair: Sentence1: {samp[0]}, Sentence2: {samp[1]}.")

    for j in range(len(sample_list)):
        messages = [
        {"role": "system", "content": f"You are a classification system trained to classify pairs of statements into 'self-contradiction' and 'non-contradiction'. You can always assume that the statements are uttered by the same speaker. Only return the label.{prime_prompt}"},
        {"role": "user", "content": sample_list[j]},
        ]
        prompt = pipeline.tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=True
        )
        terminators = [
        pipeline.tokenizer.eos_token_id,
        pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
        ]
        outputs = pipeline(
        prompt,
        max_new_tokens=10,
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.5,
        top_p=0.9,
        )
        preds[i].append(outputs[0]["generated_text"][len(prompt):])

    out_df = pd.DataFrame()
    out_df["ID"] = sample_df["ID"]
    out_df["sample"] = samples
    out_df["prediction"] = preds[i]
    out_df["gold_label"] = sample_labels
    
    out_df.to_csv(f"Llama3-8B_{sent_type}_{samp_num}_{seed}_3.tsv", sep="\t", index=True)
    

/usr/local/jupyterhub/lib64/python3.10/site-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


0 4
4


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for